# 03 — Quality & Gate

GX suites, gate 98% critical 100%, freshness 25h/26h, quarantine replay.

In [1]:
import pathlib
# suites
import glob, pathlib, json
print(glob.glob(str(pathlib.Path("gx/expectations/*.json") if pathlib.Path("gx").exists() else pathlib.Path("../gx/expectations/*.json"))))
print(glob.glob(str(pathlib.Path("gx/checkpoints/*.yml") if pathlib.Path("gx").exists() else pathlib.Path("../gx/checkpoints/*.yml"))))


['..\\gx\\expectations\\bronze.json', '..\\gx\\expectations\\gold.json', '..\\gx\\expectations\\silver.json']
['..\\gx\\checkpoints\\bronze.yml', '..\\gx\\checkpoints\\gold.yml', '..\\gx\\checkpoints\\silver.yml']


In [2]:
import pathlib
from dotenv import load_dotenv; load_dotenv(dotenv_path=pathlib.Path(".env") if pathlib.Path(".env").exists() else pathlib.Path("../.env"))
# live dq / freshness
import os; from sqlalchemy import create_engine, text
host=os.getenv("POSTGRES_HOST","localhost"); port=os.getenv("POSTGRES_PORT","5433")
user=os.getenv("POSTGRES_STREAMLIT_READER_USER","streamlit_reader"); pw=os.getenv("POSTGRES_STREAMLIT_READER_PASSWORD")
db=os.getenv("POSTGRES_WAREHOUSE_DB","banking_dw")
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
with eng.connect() as c:
    print(list(c.execute(text("SELECT layer, table_name, status, pass_pct FROM ops.dq_results ORDER BY checked_at DESC LIMIT 5")).fetchall()))
    print(list(c.execute(text("SELECT table_name, freshness_seconds FROM ops.freshness_metrics ORDER BY checked_at DESC LIMIT 5")).fetchall()))


[('gold', 'fct_transactions', 'pass', 100.0), ('gold', 'dim_customer', 'pass', 100.0), ('silver', 'transactions', 'pass', 100.0), ('silver', 'transactions', 'pass', 100.0), ('gold', 'fct_transactions', 'pass', 100.0)]
[('fct_card_transactions', 0), ('fct_transactions', 0), ('dim_account', 0), ('dim_customer', 0), ('dim_branch', 0)]


In [3]:
# gate calc
import os
gate = 98.0
score = 0.99
print(score*100 >= gate)
# critical must be 100%


True


In [4]:
# chart: dq pass_pct trend
import plotly.express as px, pandas as pd
from sqlalchemy import create_engine, text
eng=create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}")
df = pd.read_sql(text("SELECT checked_at, pass_pct, severity FROM ops.dq_results ORDER BY checked_at"), eng)
if not df.empty:
    fig = px.scatter(df, x="checked_at", y="pass_pct", color="severity")
    fig.update_layout(title="DQ pass_pct over time")
    fig.show()
else:
    print("no dq_results yet")
